# Train a small DreaMS model from scratch (Google Colab, GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Victoria125/priority-wur/blob/dreams-molecular-families/dreams_molecular_families/colab_train_from_scratch.ipynb)

**What this notebook does**
1. Installs DreaMS on a Colab GPU.
2. Downloads 500,000 public spectra (GeMS, from the DreaMS authors).
3. Trains a **smaller DreaMS model from zero** (self-supervised: it hides some peaks and learns to guess them, like DreaMS).
4. Compares our model with the published DreaMS on the actinobacteria data of this project.

**Why smaller?** The published DreaMS used 8 GPUs for about 24 hours and millions of spectra. Colab gives one GPU for a few hours and about 12 GB of memory. Our model has 4 layers instead of 7 and half the width.

**Before you start:** menu *Runtime → Change runtime type → T4 GPU*. Then run the cells from top to bottom.

**If Colab disconnects:** the model is saved in your Google Drive every 500 steps. Run all cells again: training continues from the last saved model.

Code: `scripts/train_from_scratch.py`, `scripts/embed_ssl.py`, `scripts/compare_models.py`. DreaMS commit `dbec3a0`.

## 1. Settings

In [ ]:
# Change these if you want.
HOURS = 3            # training time in this session (Colab free sessions last up to about 12 hours)
QUICK_TEST = False   # True = 50,000 spectra and 10 minutes, only to check that everything works
MODEL_NAME = "scratch_small"

## 2. Check the GPU

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU. Menu: Runtime -> Change runtime type -> T4 GPU, then run again.")
print("GPU:", gpu.stdout.strip())

## 3. Connect Google Drive

The trained model is saved here, so it is not lost when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
OUT = Path("/content/drive/MyDrive/priority_dreams") / (MODEL_NAME + ("_quicktest" if QUICK_TEST else ""))
OUT.mkdir(parents=True, exist_ok=True)
print("Model folder:", OUT)

## 4. Get the code and install DreaMS (about 10 minutes)

Colab uses Python 3.12, and DreaMS needs Python 3.11, so DreaMS gets its own environment (made with `uv`).

In [ ]:
%cd /content
![ -d priority-wur ] || git clone -q -b dreams-molecular-families https://github.com/Victoria125/priority-wur.git
%cd /content/priority-wur/dreams_molecular_families
!git pull -q
![ -d .DreaMS ] || (git clone -q https://github.com/pluskal-lab/DreaMS.git .DreaMS && git -C .DreaMS checkout -q dbec3a0b514a99e5056cfccde4559fda8cfe8129)
!pip install -q uv
![ -x .dreams_env/bin/python ] || uv venv -q .dreams_env --python 3.11
!uv pip install -q --python .dreams_env/bin/python -e ./.DreaMS mhfp==1.9.6
PY = "/content/priority-wur/dreams_molecular_families/.dreams_env/bin/python"
!{PY} -c "import torch, dreams; print('DreaMS ready. GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

## 5. Download the training spectra

Random samples of GeMS, the data DreaMS was trained on: 500,000 spectra (2.2 GB), or 50,000 (0.2 GB) for the quick test.

In [ ]:
GEMS_FILE = "GeMS_A1_DreaMS_rand50k.hdf5" if QUICK_TEST else "GeMS_A1_DreaMS_rand500k.hdf5"
DATA = Path("/content/gems") / GEMS_FILE
DATA.parent.mkdir(exist_ok=True)
if not DATA.exists():
    url = f"https://huggingface.co/datasets/roman-bushuiev/GeMS/resolve/main/data/GeMS_A/{GEMS_FILE}"
    !wget -q -O {DATA} {url}
print(DATA, round(DATA.stat().st_size / 1e9, 2), "GB")

## 6. Train

The progress bar shows the steps. The model is saved every 500 steps in your Drive. If the session stops, run all cells again and training continues where it stopped.

In [ ]:
hours = 10 / 60 if QUICK_TEST else HOURS
!{PY} scripts/train_from_scratch.py --data {DATA} --out {OUT} --size small --hours {hours}

## 7. Training loss

The loss should go down. It is the error of the model when it guesses the hidden peaks.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
logs = sorted((OUT / "log").glob("version_*/metrics.csv"))
loss = pd.concat([pd.read_csv(f) for f in logs])
loss = loss.dropna(subset=["Train loss"]).sort_values("step") if "Train loss" in loss else loss
plt.figure(figsize=(7, 3.5))
plt.plot(loss["step"], loss["Train loss"], lw=0.8)
plt.xlabel("training step"); plt.ylabel("train loss"); plt.title("Training loss")
plt.show()
print(open(OUT / "training_info.json").read())

## 8. Embeddings of our actinobacteria spectra

For every dataset of the project (protocol spectra), two self-supervised models are read the same way: the published DreaMS before fine-tuning (`dreams_ssl`, downloaded once, 1.4 GB) and our model.

In [ ]:
RUNS = ["duncan_salinispora", "crusemann_salstrep", "chevrette_insect", "dorrestein_salbus", "tahlan_streptomyces"]
for run in RUNS:
    !{PY} scripts/embed_ssl.py --run {run} --name dreams_ssl
    !{PY} scripts/embed_ssl.py --run {run} --name {MODEL_NAME} --ckpt {OUT}/final.ckpt

## 9. Compare our model with the published DreaMS

Same tests as in the main notebook: family recovery (A1), correlation with chemical structure (B1) and analog search (B2). An interval that contains 0 means no detectable difference.

In [ ]:
!{PY} scripts/compare_models.py --models {MODEL_NAME}
from IPython.display import Markdown, display
display(Markdown(open("model_comparison/RESULTS_model_comparison.md").read()))

## 10. Keep the results

The comparison and the embeddings are copied to your Drive, next to the model. Download the folder `model_comparison` from Drive if you want to add it to the GitHub repository.

In [ ]:
import shutil
results = OUT / "results"
results.mkdir(exist_ok=True)
shutil.copytree("model_comparison", results / "model_comparison", dirs_exist_ok=True)
for run in RUNS:
    for f in Path("runs", run, "data").glob(f"*{MODEL_NAME}*_embeddings_protocol.*"):
        shutil.copy(f, results / f"{run}__{f.name}")
    for f in Path("runs", run, "data").glob("dreams_ssl_embeddings_protocol.*"):
        shutil.copy(f, results / f"{run}__{f.name}")
print("Saved in", results)